In [0]:
use catalog identifier (:use_catalog)

In [0]:
-- Get the Start Date for the Tagging Process
-- We want to default to the last date in the existing data set and NOT re-process any calls that already exist in the target table. 
-- This date is INCLUSIVE only if a call on this day does not exist in the target table. 
declare or replace variable start_date date;
set variable start_date = coalesce(try_cast(:sae_start_date as date), (select date(max(`time`)) from [schema_name].experience_risk_identifier));

In [0]:
-- Get the End Date for the CoPilot AI Tagging Process
-- We want to default to today
-- This date is EXCLUSIVE - it will NOT be included in the data set
declare or replace variable end_date date;
set variable end_date = coalesce(try_cast(:sae_end_date as date), current_date);

In [0]:
-- Note the start date and end date
select
  session.start_date,
  session.end_date
;

In [0]:
CREATE OR REPLACE TABLE [schema_name].experience_risk_identifier_stg
TBLPROPERTIES ('delta.columnMapping.mode' = 'name') AS
SELECT
    `id`
  , `time`
  , to_json(transcript_list) as `transcript`,
  ai_query(
    '[llm-model-name]',
    CONCAT(
      '## Role & Objective

You are a churn-risk detection analyst for [Company]\'s Customer Success team. You read a single customer call transcript and identify whether any of **three specific renewal risks** are present, based strictly on what is said in the transcript.

You detect ONLY these three risks. Do not flag anything else.

1. **Support Issues or Poor Experience**
2. **Product Experience Issues or Bugs**
3. **Adoption Gap - Customer Has Not Achieved Outcomes** 


### Core rules

- **Ground all judgments strictly in transcript text.** Flag a risk only when explicit statements in the transcript support it. Never infer risk from silence, tone, pauses, unstated assumptions, or external context.
- **Require a verbatim quote for every flag.** Extract an exact, word-for-word quote from the transcript to justify every flagged risk. If you cannot extract a direct quote, do not flag the risk. No quote = no flag.
- **Validate speaker authority before flagging.** Evaluate statements spoken directly by the customer, or direct customer quotes restated verbatim by the account team. Partner-reported factual observations about the customer (e.g., deployment status, product fit) are valid evidence when the partner has direct operational knowledge; cap confidence at Medium. Ignore internal rep speculation, sales hypotheses, or rep-generated worry.
- **Include POV and pilot evaluations.** These risks apply to prospects in active evaluations, not only existing renewals. A failed POV is a conversion risk — flag it using the same categories and evidence standards.
- **Evaluate each risk independently.** Scan for all qualifying risks without artificial caps. A single transcript may yield zero, one, or multiple flags based on what the quotes support.
- **Default to an empty response when no risks exist.** Return an empty result when a call contains no qualifying evidence. Never force, guess, or approximate a match—clean calls are expected.
- **Map quotes precisely to risk category definitions.** Categorize risks using these explicit functional boundaries. Assign multiple risk categories when a single statement contains evidence of more than one experience risk (e.g., a product bug that also caused a poor support experience).

### Risk definitions and detection signals

**1. Support Issues or Poor Experience**
The customer has had a poor service experience affecting their view of the [Company] relationship.
- High support ticket volume with slow resolution time
- Recurring technical issues, the same problem has been escalated multiple times
- Customer has escalated support dissatisfaction to the CSM, TM, or CS Manager
- Customer references repeated negative interactions with support or expresses that their overall support experience has been poor
- Customer\'s team is spending significant time working around known bugs or limitations
*Do NOT flag:* A customer mentioning a single routine support interaction, asking a how-to question, or requesting normal product guidance. A single open ticket does not constitute a support experience risk — the signal requires evidence of repeated, unresolved, or escalated support dissatisfaction. An issue that was fully resolved during the call itself is not a support risk.
*Disambiguation:* This is about **service experience** — the quality of support interactions and responsiveness. Product defects without support complaints = Product Experience Issues. Unmet outcomes without support complaints = Adoption Gap.

**2. Product Experience Issues or Bugs**
The platform has not delivered reliable, accurate, or stable behavior — the core product promise is that [Company] data can be trusted for security decisions.
- Customer reports a specific bug or product defect affecting their workflow
- A product update has broken an existing configuration or integration
- Customer\'s scanning results are inconsistent or producing false positives at an unacceptable rate
- A feature the customer relies on has been deprecated or significantly changed
- Customer\'s integration (ServiceNow, Jira) has broken following a platform update
- Customer has lost confidence in the accuracy of the [Company] products
*Do NOT flag:* A customer asking a routine how-to question, requesting help with a new feature, or reporting a minor cosmetic issue that does not affect their workflow or data accuracy. Normal feature requests or enhancement suggestions are not product experience risks. A single newly-opened ticket for a non-blocking issue with no history of recurrence is not a product experience risk.
*Disambiguation:* This is about **product reliability and accuracy** — whether the platform delivers on its core promise. Poor support responsiveness without product defects = Support Issues. Unmet business outcomes without product defects = Adoption Gap.

**3. Adoption Gap — Customer Has Not Achieved Outcomes**
The customer purchased [Company] with an expected outcome that was not achieved — whether due to the customer\'s own bandwidth constraints or gaps in adoption enablement.
Detection signals (flag when the transcript mentions any of these):
-  Customer reaches renewal without measurable improvement in their security program
- Success plan milestones were not achieved — no concrete outcomes to point to
- Customer says "we haven\'t really gotten value out of this"
- Customer describes their usage or adoption as stagnant — no progress since deployment
- Customer compares what they were told at purchase to what they have experienced
- Leadership asks "what have we gotten for this investment?" or questions whether the spend is justified

*Do NOT flag:* A customer asking about a feature they haven\'t explored yet, requesting training on a new module, or discussing future expansion plans. Normal learning curves and feature discovery are not adoption gaps — the signal requires evidence that the customer has failed to achieve the outcomes they expected from their [Company] investment.
*Disambiguation:* This is about **outcome achievement** — whether the customer has realized the value they expected from their purchase. Poor support experience without unmet outcomes = Support Issues. Product defects without unmet outcomes = Product Experience Issues.

### Output format

Return a JSON object with the following keys:

- **`Is_CSM_Renewal_Risk_Present`** (boolean) — `true` if at least one of the three risks is present in the transcript; `false` if none qualify.
- **`CSM_Renewal_Risk_Summary`** (string) — a concise summary of the context in which the risk(s) were captured: what was happening in the conversation that surfaced them. If no risks are present, return an empty string.
- **`CSM_Renewal_Detected_Risks`** (array) — one entry per flagged risk; an empty array when none qualify. Each entry contains:
  - **`CSM_Renewal_Risk_Category`** — exactly one of: `Support Issues or Poor Experience`, `Product Experience Issues or Bugs`, `Adoption Gap - Customer Has Not Achieved Outcomes`.
  - **`CSM_Renewal_Risk_Reasoning`** — 1-3 sentences. State the observation and which detection signal it maps to. Factual; do not overstate certainty.
  - **`CSM_Renewal_Risk_Key_Verbatim_Quote`** — verbatim from the transcript. Do not paraphrase. You may stitch together non-contiguous fragments if multiple supporting quotes exist.
  - **`CSM_Renewal_Risk_Strength`** — a judgment of how severe the captured risk is: how much threat the detected risk(s) pose to the renewal. One of `N/A`, `Low`, `Medium`, `High`:
  - **Low** — a minor or isolated issue being actively addressed; the customer expresses understanding and the concern does not affect their core workflow. Typical: a single recent bug with a fix in progress, a one-time service delay.
  - **Medium** — clear frustration but the customer remains engaged and has not indicated intent to leave. Typical: multiple unresolved tickets over weeks, a workflow forced into workarounds, unmet milestones without alternative evaluation.
  - **High** — requires at least one of: customer mentions evaluating alternatives, leadership questions the investment, issue unresolved 60+ days, customer uses language like "no confidence" or "of no use," or project/adoption is fully stalled.
  - Rate conservatively and ground it in what was actually said. When no risks are detected, the level is `N/A`.
  - **`CSM_Renewal_Risk_Confidence`** - A judgement of how confident you are in your assessment of this risk mention. One of "Low","Medium","High".


### Example

Transcript excerpt:
> **Customer:** We\'ve had three critical support tickets open for over a month now and nobody seems to be able to resolve them. Every time we follow up we get a different person.
> **Customer:** And the scan results on our Linux servers keep showing false positives. We\'re spending hours every week manually filtering those out.
> **Customer:** Honestly, we bought [Company] to get better visibility into our vulnerability posture, but we\'re still running the same manual spreadsheet process we had before. We haven\'t really gotten value out of this yet.

Output:
```json
{
  "Is_CSM_Renewal_Risk_Present": true,
  "CSM_Renewal_Risk_Summary": "During a quarterly review the customer expressed frustration with unresolved support tickets, inaccurate scan results producing false positives, and a lack of measurable progress toward their security program goals.",
  "CSM_Renewal_Detected_Risks": [
    {
      "CSM_Renewal_Risk_Category": "Support Issues or Poor Experience",
      "CSM_Renewal_Risk_Reasoning": "The customer reports three critical tickets open for over a month with no resolution and inconsistent support contacts, matching the \'high support ticket volume with slow resolution time\' and \'recurring technical issues escalated multiple times\' detection signals.",
      "CSM_Renewal_Risk_Key_Verbatim_Quote": "We\'ve had three critical support tickets open for over a month now and nobody seems to be able to resolve them. Every time we follow up we get a different person.",
      "CSM_Renewal_Risk_Strength": "High",
      "CSM_Renewal_Risk_Confidence": "High"
    },
    {
      "CSM_Renewal_Risk_Category": "Product Experience Issues or Bugs",
      "CSM_Renewal_Risk_Reasoning": "The customer reports persistent false positives on Linux server scans requiring hours of manual filtering, matching the \'scanning results are inconsistent or producing false positives at an unacceptable rate\' detection signal.",
      "CSM_Renewal_Risk_Key_Verbatim_Quote": "The scan results on our Linux servers keep showing false positives. We\'re spending hours every week manually filtering those out.",
      "CSM_Renewal_Risk_Strength": "Medium",
      "CSM_Renewal_Risk_Confidence": "High"
    },
    {
      "CSM_Renewal_Risk_Category": "Adoption Gap - Customer Has Not Achieved Outcomes",
      "CSM_Renewal_Risk_Reasoning": "The customer explicitly states they have not gotten value and are still using manual processes, matching the \'customer says we haven\'t really gotten value out of this\' and \'customer describes their usage or adoption as stagnant\' detection signals.",
      "CSM_Renewal_Risk_Key_Verbatim_Quote": "We bought [Company] to get better visibility into our vulnerability posture, but we\'re still running the same manual spreadsheet process we had before. We haven\'t really gotten value out of this yet.",
      "CSM_Renewal_Risk_Strength": "High",
      "CSM_Renewal_Risk_Confidence": "High"
    }
  ]
}
```

## Transcript
',
      to_json(`transcript_list`)
    ),
    responseFormat => '{
  "type": "json_schema",
  "json_schema": {
    "name": "csm_at_risk_renewal_extraction",
    "schema": {
      "type": "object",
      "properties": {
  "Is_CSM_Renewal_Risk_Present": {"type": "boolean"},
  "CSM_Renewal_Risk_Summary": {"type": "string"},
  "CSM_Renewal_Detected_Risks": {
    "type": "array",
    "items": {
      "type": "object",
      "properties": {
        "CSM_Renewal_Risk_Category": {"type": "string", "enum": ["Support Issues or Poor Experience","Product Experience Issues or Bugs","Adoption Gap - Customer Has Not Achieved Outcomes"]},
        "CSM_Renewal_Risk_Reasoning": {"type": "string"},
        "CSM_Renewal_Risk_Key_Verbatim_Quote": {"type": "string"},
        "CSM_Renewal_Risk_Strength": {"type": "string", "enum": ["Low","Medium","High"]},
        "CSM_Renewal_Risk_Confidence": {"type": "string", "enum": ["Low","Medium","High"]}
      },
      "required": ["CSM_Renewal_Risk_Category", "CSM_Renewal_Risk_Reasoning", "CSM_Renewal_Risk_Key_Verbatim_Quote", "CSM_Renewal_Risk_Strength", "CSM_Renewal_Risk_Confidence"],
      "additionalProperties": false
    }
  }
},
"required": ["Is_CSM_Renewal_Risk_Present", "CSM_Renewal_Risk_Summary", "CSM_Renewal_Detected_Risks"],


      "additionalProperties": false
    },
    "strict": true
  }
}',
    modelParameters => named_struct('max_tokens', 8192)
  ) AS response
FROM `[catalog_name]`.`[schema_name]`.`call_details`
WHERE `transcript_text` IS NOT NULL
and array_size(filter(joinedParticipants, x -> x.email not ilike '%[company].com%' OR x.email is null)) > 0
and date(`time`) >= '2026-09-05'
and id not in (select id from [schema_name].experience_risk_identifier)



In [0]:
MERGE INTO [schema_name].experience_risk_identifier AS target
USING (
  SELECT *
  FROM [schema_name].experience_risk_identifier_stg
) AS source
  ON target.id = source.id
WHEN NOT MATCHED BY TARGET THEN INSERT *
;

In [0]:
-- Explode the detected risks array into one row per risk for easy review
CREATE OR REPLACE TABLE [schema_name].experience_risk_identifier_exploded AS
SELECT
    t.`id`
  , t.`time`
  , t.response:Is_CSM_Renewal_Risk_Present::boolean AS is_risk_present
  , t.response:CSM_Renewal_Risk_Summary::string AS risk_summary
  , r:CSM_Renewal_Risk_Category::string AS risk_category
  , r:CSM_Renewal_Risk_Strength::string AS risk_strength
  , r:CSM_Renewal_Risk_Confidence::string AS risk_confidence
  , r:CSM_Renewal_Risk_Reasoning::string AS risk_reasoning
  , r:CSM_Renewal_Risk_Key_Verbatim_Quote::string AS risk_verbatim_quote
  , t.transcript
FROM [schema_name].experience_risk_identifier t
LATERAL VIEW explode(parse_json(t.response):CSM_Renewal_Detected_Risks::array<variant>) e AS r